In [ ]:
!pip -q install python-docx openai
import os
import re
from docx import Document
from google.colab import files
from openai import OpenAI

MODEL_ID = "openai/gpt-oss-20b"
BASE_URL = "https://router.huggingface.co/v1"

In [ ]:
os.environ["HF_TOKEN"] = "REMOVED_TOKEN"
if not os.environ["HF_TOKEN"].strip():
    raise ValueError("HF_TOKEN is missing")

In [ ]:
client = OpenAI(
    base_url=BASE_URL,
    api_key=os.environ["HF_TOKEN"],
)

In [ ]:
uploaded = files.upload()

docx_files = [f for f in uploaded.keys() if f.lower().endswith(".docx")]
if not docx_files:
    raise FileNotFoundError("No DOCX uploaded. Upload your template DOCX file.")

DOCX_PATH = docx_files[0]
print("Using DOCX:", DOCX_PATH)

In [ ]:
def is_heading(line: str) -> bool:
    line = (line or "").strip()
    if len(line) < 6:
        return False
    if re.fullmatch(r"[A-Z0-9/ &\(\)\.,]+", line) and sum(c.isalpha() for c in line) >= 5:
        return True
    return False


def split_doc_into_sections(doc: Document):
    sections = []
    current_title = None
    current_lines = []

    for p in doc.paragraphs:
        text = (p.text or "").strip()

        if text == "":
            if current_title is not None:
                current_lines.append("")
            continue

        if is_heading(text):
            if current_title is not None:
                body = "\n".join(current_lines).strip()
                if len(body) > 50:
                    sections.append({"title": current_title, "text": body})
            current_title = text.strip()
            current_lines = []
        else:
            if current_title is not None:
                current_lines.append(p.text)

    if current_title is not None:
        body = "\n".join(current_lines).strip()
        if len(body) > 50:
            sections.append({"title": current_title, "text": body})

    return sections


doc = Document(DOCX_PATH)
sections = split_doc_into_sections(doc)

print("Templates found:", len(sections))
print("First few titles:")
for s in sections[:5]:
    print(" ", s["title"])

In [ ]:
print("Paste the student email. Press Enter on an empty line when done.\n")

lines = []
while True:
    line = input()
    if line == "":
        break
    lines.append(line)

USER_EMAIL = "\n".join(lines).strip()
if not USER_EMAIL:
    raise ValueError("No email entered")

print("\nEmail received.")

In [ ]:
def model_pick_from_group(email_text: str, idx_group: list[int], max_chars_per_template: int = 1600) -> int:
    blocks = []
    for pos, idx in enumerate(idx_group, start=1):
        t = sections[idx]
        body = t["text"]
        if len(body) > max_chars_per_template:
            body = body[:max_chars_per_template] + "\n[truncated]"
        blocks.append(
            f"Option {pos}\nTitle:\n{t['title']}\nTemplate:\n{body}"
        )

    prompt = f"""
Pick the single best matching template for the user email.
Match by meaning and situation.
Return only one number from 1 to {len(idx_group)}.

User email:
{email_text}

Templates:
{chr(10).join(blocks)}
""".strip()

    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": f"Return only one number from 1 to {len(idx_group)}. No other text."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=10,
    )

    out = (completion.choices[0].message.content or "").strip()
    m = re.search(r"\d+", out)
    if not m:
        return 0

    pick = int(m.group(0)) - 1
    if pick < 0:
        pick = 0
    if pick >= len(idx_group):
        pick = len(idx_group) - 1
    return pick


def tournament_best(email_text: str, batch_size: int = 5) -> int:
    remaining = list(range(len(sections)))

    while len(remaining) > 1:
        winners = []
        for start in range(0, len(remaining), batch_size):
            group = remaining[start:start + batch_size]
            if len(group) == 1:
                winners.append(group[0])
                continue

            chosen_pos = model_pick_from_group(email_text, group)
            winners.append(group[chosen_pos])

        remaining = winners

    return remaining[0]


best_index = tournament_best(USER_EMAIL, batch_size=5)
print("Chosen template title:", sections[best_index]["title"])

print("\n================ SELECTED TEMPLATE ================\n")
print(sections[best_index]["text"])
print("\n===================================================\n")